# Think-Vetor: Raciocínio Latente Contínuo e TV-DSL (Linguagem para Vetores)

Bem-vindo ao **Playground Cognitivo Unificado do Think-Vetor**!

Este notebook representa o ápice da nossa pesquisa em arquiteturas cognitivas baseadas em Micro-LLMs. Ele foi redesenhado para permitir a você realizar o **treinamento expresso do modelo de 0.5B (QLoRA) na GPU grátis do Colab** em cerca de 10-15 minutos, rodar **testes em lote automatizados de 50 perguntas com relatórios estéticos de performance** imediatos na própria nuvem e experimentar a inovação científica da **Think-Vetor DSL (TV-DSL)**.

### 🧠 O que é a Think-Vetor DSL (TV-DSL)?
Enquanto as LLMs probabilísticas tradicionais sofrem com alucinações e erros ao fazer cálculos matemáticos exatos no pensamento livre, a **TV-DSL** introduz uma mini-linguagem lógica determinística e segura escrita pelo modelo dentro de sua tag `<thought>` (ex: `[TV-DSL: multiply(432, 78)]`). Durante a inferência, nosso interpretador embutido no loop intercepta essa instrução, calcula-a deterministicamente no Python e injeta de volta o valor exato no contexto vetorial da LLM (`-> [RESULT: 33696]`). A cadeia de pensamento da LLM passa a funcionar como uma fita Turing-completa de leitura e escrita matemática exata!

## 1. Configuração do Ambiente e Instalação Expresso

A célula abaixo clonará automaticamente o repositório oficial do projeto no ambiente do Google Colab, adicionará o Python Path correto e instalará de forma silenciosa as dependências necessárias para carregar os pesos em quantização de 4-bits com LoRA.

In [ ]:
# 1. Configurar repositório e instalar dependências do ecossistema Hugging Face
import os
import sys

if not os.path.exists("crom-microllm-think-vetor"):
    print("[INFO] Clonando repositório oficial do projeto...")
    !git clone https://github.com/MrJc01/crom-microllm-think-vetor.git
    %cd crom-microllm-think-vetor
else:
    print("[INFO] Repositório já configurado.")
    %cd crom-microllm-think-vetor

# Configurar Python Path
sys.path.append(os.getcwd())
print(f"Diretório de trabalho: {os.getcwd()}")

print("\n[INFO] Instalando bibliotecas otimizadas de fine-tuning (Transformers/PEFT)...")
!pip install -q transformers peft bitsandbytes accelerate trl safetensors
print("[SUCESSO] Ambiente configurado com sucesso!")

## 2. Treinamento Expresso (QLoRA 4-bit) de 0.5B na GPU T4

Esta célula dispara o ajuste fino supervisionado do modelo base super leve de **0.5B (Qwen2.5-0.5B-Instruct)** utilizando a técnica de quantização NF4-bits (QLoRA) com acoplamento de adaptadores LoRA. 

### Por que 0.5B?
1. **Velocidade de Treino**: O fine-tuning consome apenas 5.2 GB de VRAM e conclui em apenas **10 a 15 minutos** na GPU T4 grátis do Colab.
2. **Velocidade de Inferência Local**: O modelo gerará respostas de **5 a 10 tokens por segundo** no seu CPU de 2 núcleos físicos local, fornecendo um chat fluido e sem travar os seus 12GB de RAM física (consome apenas ~1.2 GB de RAM).

In [ ]:
# 2. Executar o Treinamento QLoRA do Think-Vetor 0.5B na GPU T4
# Ajuste fino em 4-bits do modelo base leve de 0.5B (988 MB).

print("=== INICIANDO TREINAMENTO EXPRESSE DE 0.5B NO COLAB ===")
!python train_conversational_1b.py \
    --model_id "Qwen/Qwen2.5-0.5B-Instruct" \
    --epochs 3 \
    --num_samples 1200 \
    --batch_size 4 \
    --lr 2e-4 \
    --out_dir "checkpoints/think_vetor_05b_lora"

## 3. Bateria de Testes Automatizada (50 Perguntas) com Relatório Estético

Após o término do ajuste fino, esta célula carrega os pesos dos adaptadores recém-treinados e submete o modelo a uma **bateria de 50 perguntas desafiadoras** em 4 domínios diferentes (Chat/Saudação, Lógica de Transitividade, Aritmética Descritiva e Computação Pura com TV-DSL).

O script avalia a latência média, a conformidade de halting do CoT XML, os disparos da TV-DSL e gera um **Relatório Markdown Estético de Performance** para auditoria visual instantânea na própria tela do Colab!

In [ ]:
# 3. Executar Bateria de Testes Automatizada (50 Perguntas)
# Avaliação estrita de Acurácia, XML Halting, Latência e acionamentos da TV-DSL.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from src.batch_evaluator import BatchEvaluator

# Configurações do modelo
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_dir = "checkpoints/think_vetor_05b_lora"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("[INFO] Carregando modelo base e adaptadores treinados na GPU do Colab...")
tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    device_map="auto" if device.type == "cuda" else None,
    trust_remote_code=True
)
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

# Inicializar o avaliador em lote
evaluator = BatchEvaluator(model, tokenizer, device)

# Rodar os testes e exibir o relatório
report_md = evaluator.evaluate_all()

# Exibir relatório estilizado diretamente no Colab
from IPython.display import display, Markdown
display(Markdown(report_md))

# Salvar o relatório localmente para auditoria
with open("relatorio_performance_05b.md", "w", encoding="utf-8") as f:
    f.write(report_md)
print("[SUCESSO] Relatório salvo em: relatorio_performance_05b.md")

## 4. Playground Interativo com Velocidade de GPU na Nuvem

Converse em tempo real com a LLM sintonizada na GPU acelerada da nuvem e assista à ativação da TV-DSL resolvendo seus cálculos de forma infalível debaixo do capô! 

### Dica de prompt:
*   *"quanto é 432 vezes 78?"*
*   *"calcule 25 elevado a 3"*
*   *"Alice is older than Bob. Bob is older than Charlie. Who is older, Alice or Charlie?"*

In [ ]:
# 4. Chat Interativo com o Think-Vetor 0.5B + TV-DSL na Nuvem
# Converse em tempo real e assista à ativação da TV-DSL nos seus cálculos!

print("="*70)
print("     PLAYGROUND INTERATIVO COGNITIVO - THINK-VETOR CHAT 0.5B (GPU)     ")
print("="*70)
print("Digite 'sair' ou 'exit' para encerrar a conversa.")
print("-"*70)

from src.batch_evaluator import BatchEvaluator
evaluator = BatchEvaluator(model, tokenizer, device)

while True:
    try:
        user_prompt = input("\nPrompt > ").strip()
        if not user_prompt:
            continue
        if user_prompt.lower() in ["sair", "exit", "quit"]:
            print("\nPlayground encerrado. Até mais!")
            break
            
        print("\nRefletindo no Espaço Latente...")
        thought, response, latency, usou_dsl = evaluator.run_inference_tv_dsl(user_prompt)
        
        print("\n" + "="*50)
        if thought:
            print("🧠 [PENSAMENTO COGNITIVO LATENTE]")
            for line in thought.split("\n"):
                print(f"  | {line}")
            print("-" * 50)
            
        print("💬 [RESPOSTA DO ASSISTENTE]")
        print(response)
        print(f"\n[Telemetria: Latência de {latency:.2f}s | Usou TV-DSL: {usou_dsl}]")
        print("="*50 + "\n")
        
    except KeyboardInterrupt:
        print("\nPlayground encerrado.")
        break